In [1]:
# CELL 1: Environment Setup & Cloud Run Memory Purge
import os
import sys
import site
import importlib

# 1. Install packages
!pip install "numpy<2.0" "scipy<1.13" bitsandbytes --force-reinstall --no-cache-dir
!pip install torch==2.5.1 torchvision==0.20.1 transformers==4.46.2 datasets==3.1.0 tqdm==4.67.0 accelerate --no-cache-dir

# 2. Force Python to rescan the site-packages directory on the disk
importlib.reload(site)

# 3. Purge pre-loaded modules from active RAM
# This tricks Python into thinking these libraries were never imported, 
# forcing it to load the freshly installed versions in the next step.
modules_to_purge = ['numpy', 'scipy', 'torch', 'transformers', 'datasets', 'accelerate', 'bitsandbytes']
for module_name in list(sys.modules.keys()):
    if any(module_name.startswith(m) for m in modules_to_purge):
        del sys.modules[module_name]

# 4. Perform the actual imports
import json
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import bitsandbytes as bnb
from collections import namedtuple
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerBase, DynamicCache
from transformers.data.data_collator import pad_without_fast_tokenizer_warning
from datasets import load_dataset, Dataset as HFDataset
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

# Disable tokenizer parallelism to prevent deadlocks
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment setup complete. Fresh modules loaded successfully into RAM.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 175.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 270.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 280.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 272.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 307.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 256.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 261.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 288.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 256.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 261.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2026-04-21 05:39:46.103060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776749986.294257     107 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776749986.347441     107 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776749986.807249     107 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776749986.807288     107 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776749986.807290     107 computation_placer.cc:177] computation placer alr

Environment setup complete. Fresh modules loaded successfully into RAM.


In [2]:
import torch
import random
import numpy as np
import os

class Config:
    def __init__(self):
        # --- 1. THE QWEN ENGINE ---
        self.model_id = "Qwen/Qwen2.5-1.5B" 
        self.save_path = "./checkpoints_coconut_qwen"
        
        # --- 2. STRICT KAGGLE T4 SAFEGUARDS (15GB VRAM) ---
        self.batch_size_training = 1       
        self.gradient_accumulation_steps = 128
        self.max_seq_len = 512             
        
        self.lr = 5e-5  # Lower learning rate for 1.5B model
        self.weight_decay = 0.01
        
        # --- 3. PAPER-ALIGNED CURRICULUM ---
        self.hybrid_mode = True           
        self.c_thought = 2                 
        self.max_latent_tokens = 6         
        self.num_epochs_total = 50         
        
        # --- 4. EVALUATION & STEERING ---
        self.num_generations_per_sample = 1 
        self.generation_temperature = 0.0   
        self.alpha_sweep = [0.0, 5.0, 10.0, 20.0, 50.0]
        self.alpha_decay = 0.95  
        
        self.seed = 42
        self.bf16 = False # T4 GPUs prefer float16 over bfloat16
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

configs = Config()

def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_seed(configs.seed)
os.makedirs(configs.save_path, exist_ok=True)

In [3]:
def parse_example(ex):
    raw_answer = ex["answer"]
    if "####" in raw_answer:
        reasoning, final_ans = raw_answer.split("####")
        final_ans = final_ans.strip()
    else:
        reasoning = raw_answer
        final_ans = ""
    steps = [s.strip() for s in reasoning.split("\n") if s.strip()]
    return {"question": ex["question"], "steps": steps, "answer": final_ans}

print("Loading and splitting GSM8K dataset...")
raw_dataset = load_dataset("gsm8k", "main")

train_full = [parse_example(ex) for ex in raw_dataset["train"]]
total_samples = len(train_full)
idx_phase1_end = int(total_samples * 0.60)                 
idx_phase2_end = idx_phase1_end + int(total_samples * 0.10) 

data_phase1 = train_full[:idx_phase1_end]
data_phase2 = train_full[idx_phase1_end : idx_phase2_end]
data_phase3 = train_full[idx_phase2_end :] 
test_data = [parse_example(ex) for ex in raw_dataset["test"]]

print(f"Total Train Sub-Dataset: {total_samples}")
print(f"Phase 1 (Base Train): {len(data_phase1)}")
print(f"Phase 2 (Vector Extraction): {len(data_phase2)}")
print(f"Phase 3 (Validation): {len(data_phase3)}")
print(f"Phase 4 (Test Set): {len(test_data)}")

def get_hf_dataset(raw_data, tokenizer):
    def tokenize_sample(sample):
        q_tok = tokenizer.encode(sample["question"] + "\n", add_special_tokens=True)
        s_tok = [tokenizer.encode(s + "\n", add_special_tokens=False) for s in sample["steps"]]
        a_tok = tokenizer.encode("#### " + sample["answer"], add_special_tokens=False) + [tokenizer.eos_token_id]
        return {"question_tokenized": q_tok, "steps_tokenized": s_tok, "answer_tokenized": a_tok, "ground_truth": sample["answer"]}
    
    ds = HFDataset.from_list(raw_data)
    return ds.map(tokenize_sample, num_proc=4)

Loading and splitting GSM8K dataset...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Total Train Sub-Dataset: 7473
Phase 1 (Base Train): 4483
Phase 2 (Vector Extraction): 747
Phase 3 (Validation): 2243
Phase 4 (Test Set): 1319


In [4]:
from collections import namedtuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DynamicCache
from torch.nn import CrossEntropyLoss
import numpy as np

Outputs = namedtuple("Outputs", ["loss", "inputs_embeds", "logits", "latent_sequence"])

class Coconut(nn.Module):
    def __init__(self, base_causallm, latent_token_id, start_latent_id, end_latent_id, eos_token_id):
        super().__init__()
        self.base_causallm = base_causallm
        self.latent_token_id = latent_token_id
        self.eos_token_id = eos_token_id
        self.embedding = self.base_causallm.get_input_embeddings()

    def _process_kv(self, kv_cache, keep_len):
        if kv_cache is None: return None
        new_cache = DynamicCache()
        num_layers = len(kv_cache.key_cache) if hasattr(kv_cache, "key_cache") else len(kv_cache)
        for i in range(num_layers):
            try:
                if hasattr(kv_cache, "key_cache"):
                    k, v = kv_cache.key_cache[i], kv_cache.value_cache[i]
                else:
                    k, v = kv_cache[i]
            except:
                k, v = kv_cache[i]
            new_cache.update(k[..., :keep_len, :], v[..., :keep_len, :], layer_idx=i)
        return new_cache

    def forward(self, input_ids, attention_mask, labels=None, steering_vector=None, alpha=0.0, gamma=1.0):
        logits = []
        latent_sequence = []
        
        latent_indices = (input_ids == self.latent_token_id).nonzero()
        latent_lists = [[idx[1].item() for idx in latent_indices if idx[0] == i] for i in range(input_ids.shape[0])]
        max_n_latents = max([len(l) for l in latent_lists]) if latent_lists else 0

        inputs_embeds = self.embedding(input_ids)
        position_ids = torch.arange(0, input_ids.shape[1], dtype=torch.long, device=input_ids.device).unsqueeze(0)
        
        next_compute_range = (0, input_ids.shape[1])
        if max_n_latents > 0:
            next_compute_range = (0, latent_indices[:, 1].min().item())

        kv_cache = None

        for pass_idx in range(max_n_latents):
            curr_cache = self._process_kv(kv_cache, next_compute_range[0])
            outputs = self.base_causallm(
                inputs_embeds=inputs_embeds[:, next_compute_range[0] : next_compute_range[1]],
                attention_mask=attention_mask[:, :next_compute_range[1]],
                position_ids=position_ids[:, next_compute_range[0] : next_compute_range[1]], 
                past_key_values=curr_cache,
                output_hidden_states=True,
                use_cache=True 
            )
            logits.append(outputs.logits)

            next_end = input_ids.shape[1] if pass_idx + 1 >= max_n_latents else next_compute_range[1] + 1
            next_compute_range = (next_compute_range[1], next_end)

            hidden_states = outputs.hidden_states[-1]
            
            current_alpha = alpha * (gamma ** pass_idx)
            if steering_vector is not None and current_alpha > 0:
                sigma_l = hidden_states.std(dim=-1, keepdim=True)
                norm_dir = F.normalize(steering_vector, p=2, dim=-1).to(hidden_states.device)
                hidden_states = hidden_states + (current_alpha * sigma_l * norm_dir)

            latent_sequence.append(hidden_states.detach())
            
            kv_cache = outputs.past_key_values
            inputs_embeds = inputs_embeds.clone()

            filling_indices = [(i, l[pass_idx]) for i, l in enumerate(latent_lists) if len(l) > pass_idx]
            for batch_idx, token_idx in filling_indices:
                inputs_embeds[batch_idx, token_idx] = hidden_states[batch_idx, -1]

        final_cache = self._process_kv(kv_cache, next_compute_range[0])
        outputs = self.base_causallm(
            inputs_embeds=inputs_embeds[:, next_compute_range[0] : next_compute_range[1]],
            attention_mask=attention_mask[:, :next_compute_range[1]],
            position_ids=position_ids[:, next_compute_range[0] : next_compute_range[1]], 
            past_key_values=final_cache,
            use_cache=True 
        )
        logits.append(outputs.logits)
        logits = torch.cat(logits, dim=-2)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            # FIX: Upcast logits to float32 to prevent exponentiation overflow during CrossEntropy
            loss = CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)).to(torch.float32), shift_labels.view(-1))

        return Outputs(loss, inputs_embeds, logits, latent_sequence)

    def generate_with_latents(self, input_ids, max_new_tokens=128, temperature=0.0, steering_vector=None, alpha=0.0, gamma=1.0):
        self.eval()
        tokens = input_ids.tolist()[0]

        with torch.no_grad():
            outputs = self.forward(input_ids=input_ids, attention_mask=torch.ones_like(input_ids), steering_vector=steering_vector, alpha=alpha, gamma=gamma)
            
        mean_latent = torch.mean(torch.stack([h[:, -1, :] for h in outputs.latent_sequence]), dim=0).cpu() if outputs.latent_sequence else None
        
        faithfulness_scores = []
        if steering_vector is not None and outputs.latent_sequence and alpha > 0.0:
            for h in outputs.latent_sequence:
                sim = F.cosine_similarity(h[:, -1, :], steering_vector.unsqueeze(0), dim=-1)
                faithfulness_scores.append(sim.item())
        avg_faithfulness = np.mean(faithfulness_scores) if faithfulness_scores else 0.0

        if temperature > 0:
            scaled_logits = outputs.logits[:, -1, :] / temperature
            probs = torch.nn.functional.softmax(scaled_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
        else:
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).item()
            
        tokens.append(next_token)
        curr_input_ids = torch.tensor([tokens], device=input_ids.device)
        
        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = self.base_causallm(input_ids=curr_input_ids)
                
            if temperature > 0:
                scaled_logits = outputs.logits[:, -1, :] / temperature
                probs = torch.nn.functional.softmax(scaled_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).item()
            else:
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1).item()
                
            if next_token == self.eos_token_id: break
            tokens.append(next_token)
            curr_input_ids = torch.tensor([tokens], device=input_ids.device)

        return torch.tensor([tokens]), mean_latent, avg_faithfulness

In [5]:
from dataclasses import dataclass
from torch.nn import CrossEntropyLoss


In [6]:
!pip install peft

In [ ]:
import gc
from dataclasses import dataclass
import itertools
import bitsandbytes as bnb
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerBase
from torch.utils.data import DataLoader
from transformers.data.data_collator import pad_without_fast_tokenizer_warning
from tqdm.notebook import tqdm
from peft import LoraConfig, get_peft_model # --- NEW: LoRA Imports ---

def get_stage_info(epoch):
    if epoch < 6: return 0, False, (epoch == 0)
    elif epoch < 9: return 1, False, (epoch == 6)
    elif epoch < 12: return 2, False, (epoch == 9)
    elif epoch < 15: return 3, False, (epoch == 12)
    else: return 4, True, (epoch == 15)

def get_cot_latent_dataset(scheduled_stage, drop_remaining, base_dataset, configs, start_id, latent_id, end_id, shuffle=False):
    def process_dataset(sample):
        if len(sample["steps_tokenized"]) > 0 and configs.hybrid_mode:
            skeleton_text = sample["steps_tokenized"][0]
            remaining_steps = sample["steps_tokenized"][1:]
        else:
            skeleton_text = []
            remaining_steps = sample["steps_tokenized"]

        steps_to_drop = min(scheduled_stage, len(remaining_steps))
        
        if drop_remaining:
            kept_remaining_steps = []
            n_latent_tokens = configs.max_latent_tokens
        else:
            kept_remaining_steps = remaining_steps[steps_to_drop:]
            n_latent_tokens = steps_to_drop * configs.c_thought
            
        kept_remaining_text = list(itertools.chain.from_iterable(kept_remaining_steps))

        tokens = (sample["question_tokenized"] + skeleton_text + 
                  [start_id] + [latent_id] * n_latent_tokens + [end_id] +
                  kept_remaining_text + sample["answer_tokenized"])
        
        mask_len = len(sample["question_tokenized"]) + len(skeleton_text) + n_latent_tokens + 2
        labels = [-100] * mask_len + tokens[mask_len:]
        
        tokens = tokens[:configs.max_seq_len]
        labels = labels[:configs.max_seq_len]
        return {"input_ids": tokens, "labels": labels, "attention_mask": [1] * len(tokens)}
    
    dataset = base_dataset.map(process_dataset, remove_columns=list(base_dataset.features))
    if shuffle: dataset = dataset.shuffle(seed=configs.seed)
    return dataset

@dataclass
class MyCollator:
    tokenizer: PreTrainedTokenizerBase
    latent_id: int
    label_pad_token_id: int = -100

    def __call__(self, features):
        earliest_latent = [f["input_ids"].index(self.latent_id) for f in features if self.latent_id in f["input_ids"]]
        if earliest_latent:
            latest_earliest = max(earliest_latent)
            for feature in features:
                pad = latest_earliest - feature["input_ids"].index(self.latent_id) if self.latent_id in feature["input_ids"] else 0
                feature["input_ids"] = [self.tokenizer.pad_token_id] * pad + feature["input_ids"]
                feature["attention_mask"] = [0] * pad + feature["attention_mask"]
                if "labels" in feature: feature["labels"] = [self.label_pad_token_id] * pad + feature["labels"]
        
        labels = [f.pop("labels") for f in features] if "labels" in features[0] else None
        batch = pad_without_fast_tokenizer_warning(self.tokenizer, features, padding=True, return_tensors="pt")
        if labels:
             max_len = batch["input_ids"].shape[1]
             batch["labels"] = torch.tensor([l + [self.label_pad_token_id]*(max_len-len(l)) for l in labels])
        return batch

print("Initializing Qwen Model for Phase 1...")
model = AutoModelForCausalLM.from_pretrained(
    configs.model_id, 
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
).to(configs.device)

tokenizer = AutoTokenizer.from_pretrained(configs.model_id)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

tokenizer.add_tokens(["<|start-latent|>", "<|end-latent|>", "<|latent|>"])
latent_id, start_id, end_id = tokenizer.convert_tokens_to_ids(["<|latent|>", "<|start-latent|>", "<|end-latent|>"])
model.resize_token_embeddings(len(tokenizer))

with torch.no_grad():
    input_embeds = model.get_input_embeddings()
    init_id = tokenizer.encode("The", add_special_tokens=False)[0] 
    input_embeds.weight.data[latent_id] = input_embeds.weight.data[init_id].clone()
    if hasattr(model, 'lm_head') and model.lm_head is not None:
        model.lm_head.weight.data[latent_id] = model.lm_head.weight.data[init_id].clone()

# ====================================================================
# NEW: APPLY LORA TO FREEZE BASE WEIGHTS AND SAVE 6GB VRAM
# ====================================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    modules_to_save=["embed_tokens", "lm_head"], # Explicitly tell LoRA to train our new <|latent|> tokens
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # This will print out exactly how much memory you just saved!
# ====================================================================

coconut_model = Coconut(model, latent_id, start_id, end_id, tokenizer.eos_token_id).to(configs.device)
collator = MyCollator(tokenizer, latent_id=latent_id)
ds_phase1 = get_hf_dataset(data_phase1, tokenizer)

optimizer = None

print("Starting Phase 1 Multi-Stage COCONUT Training with BFloat16 & LoRA...")
for epoch in range(configs.num_epochs_total):
    current_stage, drop_remaining, requires_reset = get_stage_info(epoch)
    
    if requires_reset or optimizer is None:
        print(f"\n[Epoch {epoch}] Stage shifted to {current_stage}. Hard resetting AdamW Optimizer...")
        # Now passing only the drastically reduced trainable parameters to the optimizer
        optimizer = bnb.optim.AdamW8bit(filter(lambda p: p.requires_grad, coconut_model.parameters()), lr=configs.lr, weight_decay=configs.weight_decay)
        torch.cuda.empty_cache()

    train_ds = get_cot_latent_dataset(current_stage, drop_remaining, ds_phase1, configs, start_id, latent_id, end_id, shuffle=True)
    train_loader = DataLoader(train_ds, batch_size=configs.batch_size_training, collate_fn=collator, shuffle=True)

    coconut_model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{configs.num_epochs_total} | Stage {current_stage}")
    
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(configs.device)
        attention_mask = batch["attention_mask"].to(configs.device)
        labels = batch["labels"].to(configs.device)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = coconut_model(input_ids, attention_mask, labels)
            loss = outputs.loss / configs.gradient_accumulation_steps

        loss.backward()

        if (step + 1) % configs.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(coconut_model.parameters(), max_norm=1.0)
            
            optimizer.step()
            optimizer.zero_grad()
            
            # --- NEW: Aggressive Memory Clearing ---
            del outputs
            del loss
            torch.cuda.empty_cache()

        # Update tracking loss (using a detached clone to prevent graph memory leaks)
        total_loss += loss.item() * configs.gradient_accumulation_steps if 'loss' in locals() else total_loss
        pbar.set_postfix({"loss": total_loss / (step + 1)})

Initializing Qwen Model for Phase 1...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Starting Phase 1 Multi-Stage COCONUT Training with BFloat16 & LoRA...

[Epoch 0] Stage shifted to 0. Hard resetting AdamW Optimizer...


Map:   0%|          | 0/4483 [00:00<?, ? examples/s]

Epoch 1/50 | Stage 0:   0%|          | 0/4483 [00:00<?, ?it/s]

Map:   0%|          | 0/4483 [00:00<?, ? examples/s]

Epoch 2/50 | Stage 0:   0%|          | 0/4483 [00:00<?, ?it/s]

In [ ]:
print("Starting Phase 2: Global Mass Mean Shift Vector Extraction...")
coconut_model.eval()

ds_phase2 = get_hf_dataset(data_phase2, tokenizer) 
correct_latents = []
wrong_latents = []
n_latents_infer = configs.max_latent_tokens

for sample in tqdm(ds_phase2, desc="Extracting Latent Thoughts"):
    prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents_infer + "<|end-latent|>"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
    ground_truth = sample["ground_truth"].replace(",", "").strip()
    
    with torch.no_grad():
        # THE FIX: Added ', _' to catch the faithfulness score returned by the updated function
        generated_ids, mean_latent, _ = coconut_model.generate_with_latents(
            input_ids, 
            max_new_tokens=64,   
            temperature=configs.generation_temperature,
            alpha=0.0
        )
        
    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    pred = output_text.split("####")[-1].strip() if "####" in output_text else output_text.split(" ")[-1].strip()
    pred_clean = pred.replace(",", "").replace(".", "").split(" ")[0]
    
    # Strict empty-string bug fix
    is_correct = False
    if len(pred_clean) > 0:
        is_correct = (pred_clean == ground_truth)
    
    if mean_latent is not None:
        if is_correct:
            correct_latents.append(mean_latent)
        else:
            wrong_latents.append(mean_latent)

if len(correct_latents) > 0 and len(wrong_latents) > 0:
    mu_true = torch.mean(torch.stack(correct_latents), dim=0)
    mu_false = torch.mean(torch.stack(wrong_latents), dim=0)
    truth_vector = (mu_true - mu_false).to(configs.device)
else:
    print("WARNING: Missing examples. Vector will be zero.")
    truth_vector = torch.zeros((1, model.config.n_embd)).to(configs.device)

print(f"Total Correct Trajectories: {len(correct_latents)}")
print(f"Total Incorrect Trajectories: {len(wrong_latents)}")
print(f"Computed Global Truth Vector. Magnitude: {torch.norm(truth_vector).item():.6f}")

In [ ]:
print("Starting Phase 4: Full Pipeline Evaluation...")
coconut_model.eval()
eval_subset = test_data 
n_latents_infer = configs.max_latent_tokens

# --- GENERATE RANDOM NOISE VECTOR FOR COMPARISON ---
random_vector = torch.randn_like(truth_vector)
random_vector = random_vector / torch.norm(random_vector) * torch.norm(truth_vector)

# =====================================================================
# PART 1: STRUCTURAL ABLATION COMPARISONS & TOKEN COUNTS
# =====================================================================
print("\n" + "="*65)
print("PART 1: STRUCTURAL ABLATION COMPARISONS")
print("="*65)

conditions = [
    {"name": "No CoT", "prompt_type": "direct", "steering": None, "alpha": 0.0},
    {"name": "Text based CoT", "prompt_type": "text", "steering": None, "alpha": 0.0},
    {"name": "Just CCoT", "prompt_type": "latent", "steering": None, "alpha": 0.0},
    {"name": "Random Noise (α=20.0)", "prompt_type": "latent", "steering": random_vector, "alpha": 20.0}
]

comparison_results = []

for condition in conditions:
    correct = 0
    total_generated_tokens = 0
    total = len(eval_subset)
    
    for sample in tqdm(eval_subset, desc=condition['name']):
        if condition["prompt_type"] == "direct":
            prompt = sample["question"] + "\nAnswer:"
        elif condition["prompt_type"] == "text":
            prompt = sample["question"] + "\nLet's think step by step:\n"
        else: 
            prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents_infer + "<|end-latent|>"
            
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
        ground_truth = sample["answer"].replace(",", "").strip()

        with torch.no_grad():
            generated_ids, _, _ = coconut_model.generate_with_latents(
                input_ids, 
                max_new_tokens=256 if condition["prompt_type"] == "text" else 128, 
                temperature=0.0, 
                steering_vector=condition["steering"],
                alpha=condition["alpha"],
                gamma=configs.alpha_decay
            )

        # Calculate exact number of generated tokens (Output Length - Prompt Length)
        gen_token_count = generated_ids.shape[1] - input_ids.shape[1]
        total_generated_tokens += gen_token_count

        output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        
        if "####" in output_text:
            pred = output_text.split("####")[-1].strip()
        elif "Answer:" in output_text and condition["prompt_type"] == "direct":
            pred = output_text.split("Answer:")[-1].strip()
        else:
            pred = output_text.split(" ")[-1].strip()
            
        pred_clean = pred.replace(",", "").replace(".", "").split(" ")[0]
        
        # Strict correctness check
        is_correct = False
        if len(pred_clean) > 0:
            is_correct = (pred_clean == ground_truth)
        
        if is_correct:
            correct += 1

    acc = correct / total
    avg_tokens = total_generated_tokens / total
    comparison_results.append((condition["name"], acc, avg_tokens))

for name, acc, avg_tokens in comparison_results:
    print(f"Compare against {name.lower():<22}: Accuracy {acc:>6.2%} | Avg Tokens: {avg_tokens:.1f}")

# =====================================================================
# PART 2: ITI TRUTH VECTOR ALPHA SWEEP
# =====================================================================
print("\n" + "="*65)
print("PART 2: ITI TRUTH VECTOR SWEEP")
print("="*65)

alpha_sweep_results = []
baseline_correctness = [] 
total_baseline_incorrect = 0

for alpha in configs.alpha_sweep:
    correct = 0
    flips = 0
    total = len(eval_subset)
    faithfulness_list = []
    
    for idx, sample in enumerate(tqdm(eval_subset, desc=f"Sweeping Alpha={alpha}")):
        prompt = sample["question"] + "\n<|start-latent|>" + "<|latent|>" * n_latents_infer + "<|end-latent|>"
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(configs.device)
        ground_truth = sample["answer"].replace(",", "").strip()

        with torch.no_grad():
            generated_ids, _, faith = coconut_model.generate_with_latents(
                input_ids, 
                max_new_tokens=128, 
                temperature=0.0, 
                steering_vector=truth_vector if alpha > 0.0 else None,
                alpha=alpha,
                gamma=configs.alpha_decay
            )

        faithfulness_list.append(faith)
        output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        
        if "####" in output_text:
            pred = output_text.split("####")[-1].strip()
        else:
            pred = output_text.split(" ")[-1].strip()
            
        pred_clean = pred.replace(",", "").replace(".", "").split(" ")[0]
        
        is_correct = False
        if len(pred_clean) > 0:
            is_correct = (pred_clean == ground_truth)
            
        if is_correct:
            correct += 1
            
        # Record baseline for Flip Rate calculations
        if alpha == 0.0:
            baseline_correctness.append(is_correct)
            if not is_correct:
                total_baseline_incorrect += 1
        else:
            # If it was wrong at baseline, but is correct now, that is a Flip
            if not baseline_correctness[idx] and is_correct:
                flips += 1

    acc = correct / total
    flip_rate = (flips / total_baseline_incorrect) if total_baseline_incorrect > 0 else 0.0
    avg_faith = np.mean(faithfulness_list)
    
    alpha_sweep_results.append((alpha, acc, flip_rate, avg_faith))

# Print exactly formatted table
print("\nAlpha | Accuracy | Flip Rate | Faithfulness")
for alpha, acc, flip, faith in alpha_sweep_results:
    print(f"{alpha:<5.1f} | {acc:>6.2%} | {flip:>8.2%} | {faith:.4f}")

In [ ]:
import torch
import torch.nn.functional as F

def decode_vector_to_text(vector, base_model, tokenizer, top_k=15):
    print("\n" + "="*50)
    print(" LOGIT LENS: TRANSLATING TRUTH VECTOR TO TEXT")
    print("="*50)
    
    base_model.eval()
    with torch.no_grad():
        # Retrieve the unembedding matrix (LM Head)
        lm_head = base_model.get_output_embeddings()
        
        # Project the [1, hidden_size] vector into the [vocab_size] dimensional space
        logits = lm_head(vector.to(configs.device))
        
        # Convert the raw projection scores into probabilities
        probs = F.softmax(logits[0], dim=-1)
        
        # Extract the top K most probable tokens
        top_probs, top_indices = torch.topk(probs, k=top_k)
        
        print(f"Top {top_k} vocabulary tokens aligned with this vector direction:\n")
        for i in range(top_k):
            token_id = top_indices[i].item()
            # Decode the token ID back into readable English
            token_str = tokenizer.decode([token_id])
            # Escape newlines/tabs for clean terminal printing
            token_str_clean = repr(token_str) 
            prob = top_probs[i].item()
            
            print(f"{i+1:>2}. Token: {token_str_clean:<15} | ID: {token_id:<6} | Probability Mass: {prob:.4%}")

# Pass your extracted truth_vector, the raw HuggingFace model, and the tokenizer
decode_vector_to_text(truth_vector, model, tokenizer, top_k=15)